# 🤝 Bridgeway – Agent-Powered Business Assistant

**EECE 503P – C3: Build an Agent-Powered Business Assistant**

This notebook builds **Rima**, the outreach advisor of *Bridgeway – Fundraising & Communications Consultancy for NGOs* (Hamra, Beirut),
an agency that helps Lebanese NGOs reach donors, partners and volunteers.
Rima answers questions from the business documents, collects leads from interested NGOs, and records feedback and unanswered questions.

**How the agent works (one message at a time):**

```
user message ──► ROUTER (general_question / lead_collection / feedback)
                    │  adds a per-turn hint
                    ▼
             SYSTEM PROMPT = Persona + Instructions + Business Summary + Business PDF + hint
                    │
                    ▼
             AGENT LOOP  ◄──►  TOOLS (record_customer_interest, record_feedback)
                    │
                    ▼
             REFLECTION (checks the draft, revises if needed) ──► reply
```

**Sections**
1. Persona
2. Instructions
3. Tools
4. Knowledge + system prompt
5. Router
6. LLM client + agent loop
7. Reflection (bonus)
8. Gradio demo

## 🤝 Persona

The **persona** defines *who* the assistant is: its name, role, personality and tone of voice.
It does **not** contain rules about what the bot is allowed to do. Those are in **Instructions** below.
Keeping the two apart makes each one easy to read and change without breaking the other.

In [ ]:
# PERSONA = WHO the assistant is (identity, personality, tone of voice).
PERSONA = """\
You are Rima, the outreach advisor of Bridgeway – Fundraising & Communications Consultancy for NGOs in Hamra, Beirut.

Personality and tone:
- Warm, mission-driven and encouraging. You genuinely care about the causes NGOs work for.
- Professional and clear: short, friendly replies in plain language, no jargon.
- Respectful of NGOs' limited budgets: you point out the discount that applies and never upsell.
- Never salesy or pushy. You help people decide; you don't push them to buy.
- You represent the Bridgeway team (Nour, Hadi, Leila and Sami) and speak as "we" for the consultancy.
"""

## 📋 Instructions

The **instructions** define *what the assistant must do*: numbered rules, one purpose each.
They decide when to call each tool, what is off-limits (inventing facts, promising results, confirming bookings),
how discounts are applied, and how long replies should be. A short worked example at the end shows the model the exact behavior we expect.

They don't conflict with the persona: the persona sets the *voice*, and the instructions set the *behavior*.

In [ ]:
# INSTRUCTIONS = WHAT the assistant must do (numbered rules, one purpose each).
INSTRUCTIONS = """\
Follow these rules in every reply:

1. Answer only from the Business Summary and Business Profile provided below. Never invent prices, services, discounts, timelines or results.
2. Apply the discounts exactly as written: 20% off for small NGOs (annual budget under 100,000 USD), 10% off when booking 2 or more services. The discounts do not stack: apply only the bigger one. Show the calculation (e.g. 600 minus 20% = 480 USD).
3. Never promise or estimate how much money, or how many donors, partners or volunteers, a campaign will bring. We report results honestly at the end; we never guarantee them.
4. If something is not in the business information, or is not offered (for example: writing full grant proposals, buying or selling email lists, paid ads, handling or collecting donation money, outreach for political parties or for-profit companies), say so politely, and call record_feedback with the user's question.
5. When a user shows interest in a service, ask for their name and email, plus their NGO's name and the service they need if they haven't said. Ask only for what is still missing.
6. Call record_customer_interest only once you have at least a name and a valid-looking email (like name@example.org). Put the NGO name, the service and the timing in the message argument. If the tool returns an error, politely ask the user to correct that detail.
7. Never confirm a booking, a start date or a call time. Say that our team will email them to schedule the free 30-minute discovery call.
8. When a user gives feedback or a complaint, thank them and call record_feedback with their feedback.
9. Gently invite interested users to leave their name and email so our team can follow up.
10. Keep replies under about 120 words unless you are listing prices. Use short bullet lists for prices (all prices are in USD).

Worked example:
User: "I'm Maya from GreenSteps, maya@greensteps.org, we need volunteers in March."
Correct behavior:
- Call record_customer_interest(name="Maya", email="maya@greensteps.org", message="GreenSteps – volunteer drive wanted for March").
- Then reply: thank Maya, mention the volunteer drive is 300 USD (4 weeks: social posts, emails, sign-up form), mention the 20% small-NGO discount if their budget is under 100,000 USD, and say our team will email to schedule the free discovery call. Do NOT say the campaign is booked, and do NOT promise a number of volunteers.
"""

In [ ]:
# Quick check: print both blocks so we can see exactly what the model will receive.
print("=== PERSONA ===")
print(PERSONA)
print("=== INSTRUCTIONS ===")
print(INSTRUCTIONS)

## 🛠️ Tools

A **tool** is a normal Python function the model is allowed to *ask* us to run.
The model never runs code itself. It replies with a *tool call* (tool name + JSON arguments),
our code runs the function, and we send the result back so the model can write its answer.

Rima has two tools. Both live in [`tools.py`](tools.py) so the offline tests (Step 9) can import them:

| Tool | When the model calls it | Saved to |
|---|---|---|
| `record_customer_interest(email, name, message)` | Someone from an NGO is interested in a service and has given a name + valid email | `logs/leads.jsonl` |
| `record_feedback(question)` | Rima can't answer from the business info, the user asks for something we don't offer, **or** the user gives feedback/complaints | `logs/feedback.jsonl` |

Every tool **validates its input**, **prints a log line** (📝 / 💬), **appends one JSON line** with an ISO timestamp,
and returns either `{"status": "recorded", "id": ...}` or `{"status": "error", "reason": ...}`.
An error is not a crash: the model reads the `reason` and asks the user to fix that detail (Instruction 6).

### MCP-style tool definitions

We document each tool the way the **Model Context Protocol (MCP)** does: `name`, `description`, `inputSchema` (JSON Schema).
The `description` matters a lot, because it is how the model decides *when* to call the tool.

**`record_customer_interest`**

```json
{
  "name": "record_customer_interest",
  "description": "Record a lead: someone from an NGO who is interested in one of Bridgeway's services ... Call it as soon as you have at least the person's name and a valid-looking email. Do not call it for general questions without contact details.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "email":   {"type": "string", "description": "The person's email address, e.g. maya@greensteps.org"},
      "name":    {"type": "string", "description": "The person's name, e.g. Maya"},
      "message": {"type": "string", "description": "What they want: NGO name, service(s), timing"}
    },
    "required": ["email", "name", "message"]
  }
}
```

**`record_feedback`**

```json
{
  "name": "record_feedback",
  "description": "Record something the team must review. Call it when (a) the user asks about something not in the business information, (b) the user asks for a service Bridgeway does not offer, or (c) the user gives feedback, praise or a complaint.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "question": {"type": "string", "description": "The user's unanswered question or their feedback, in their words."}
    },
    "required": ["question"]
  }
}
```

The OpenAI chat API wants a slightly different wrapper (`{"type": "function", "function": {name, description, parameters}}`),
so `tools.py` writes the definitions **once** in MCP format and converts them into `OPENAI_TOOLS` automatically.
`TOOL_REGISTRY` maps each tool name to its Python function, and `run_tool(name, arguments)` runs a model's tool call safely.

In [ ]:
# Import the tools from tools.py (kept outside the notebook so tests can use them too).
import json

from tools import (
    MCP_TOOL_DEFINITIONS,  # the MCP-style docs shown above
    OPENAI_TOOLS,          # the same definitions in the format the OpenAI API expects
    TOOL_REGISTRY,         # tool name -> Python function
    run_tool,              # safely runs a tool call coming from the model
    record_customer_interest,
    record_feedback,
)

print("Registered tools:", list(TOOL_REGISTRY))
print(json.dumps(OPENAI_TOOLS[0], indent=2))

In [ ]:
# Try the tools by hand, exactly as the agent loop will later (tool name + JSON string of arguments).
# Successful calls print a log line and append to logs/leads.jsonl or logs/feedback.jsonl.

print("1) Valid lead:")
print(run_tool("record_customer_interest",
               '{"email": "maya@greensteps.org", "name": "Maya", "message": "GreenSteps – volunteer drive wanted for March"}'))

print()
print("2) Bad email -> error the model can react to:")
print(run_tool("record_customer_interest", '{"email": "maya-at-greensteps", "name": "Maya", "message": "Volunteer drive"}'))

print()
print("3) A request Rima must decline (not offered):")
print(run_tool("record_feedback", '{"question": "Can you write our EU grant proposal?"}'))

print()
print("4) A tool that doesn't exist:")
print(run_tool("book_campaign", '{}'))

## 📚 Knowledge + System Prompt

A language model only "knows" what is in its **context**. To make Rima answer from *our* business,
we paste both business documents into the **system prompt**, the hidden first message that sets the rules for the whole chat.

`knowledge.py` loads them:
- `load_business_summary()` reads `me/business_summary.txt`
- `load_business_profile()` extracts the text of `me/about_business.pdf` with `pypdf`

If a file is missing or unreadable we get a clear `KnowledgeError` instead of a confusing crash.

`build_system_prompt(route_hint)` stacks the blocks in this order:

```
PERSONA          → who Rima is
INSTRUCTIONS     → the rules
Business Summary → business_summary.txt
Business Profile → about_business.pdf
Note for this message (optional) → hint from the router (Step 5)
```

This is a simple form of *retrieval*: our documents are small (about 1,000 words), so we include all of them on every turn instead of searching for the relevant part.

In [ ]:
from knowledge import (
    KnowledgeError,
    compose_system_prompt,
    load_business_profile,
    load_business_summary,
)

# Load both documents once, when the notebook starts.
BUSINESS_SUMMARY = load_business_summary()
BUSINESS_PROFILE = load_business_profile()


def build_system_prompt(route_hint=None):
    """PERSONA + INSTRUCTIONS + both business documents (+ the router's hint for this message)."""
    return compose_system_prompt(PERSONA, INSTRUCTIONS, BUSINESS_SUMMARY, BUSINESS_PROFILE, route_hint)


system_prompt = build_system_prompt()
print(f"Summary: {len(BUSINESS_SUMMARY):,} characters · PDF: {len(BUSINESS_PROFILE):,} characters")
print(f"System prompt: {len(system_prompt):,} characters (~{len(system_prompt) // 4:,} tokens)\n")
print(system_prompt[:1200], "\n[...]")

In [ ]:
# Sanity check: every price in business_data.py (our single source of truth) must appear in the PDF
# that the model reads. If someone edits a price in one place only, this cell tells us.
from business_data import DISCOUNTS, PRICES, discounted_price


def all_prices(prices):
    """Yield every number inside the (nested) PRICES dict."""
    for value in prices.values():
        if isinstance(value, dict):
            yield from all_prices(value)
        else:
            yield value


missing = sorted({p for p in all_prices(PRICES) if f"${p}" not in BUSINESS_PROFILE})
rules = [f"{DISCOUNTS['small_ngo']['percent']}% off for small NGOs", f"{DISCOUNTS['bundle']['percent']}% off when booking"]
print("Prices missing from the PDF:", missing or "none ✅")
print("Discount rules in the PDF:", "yes ✅" if all(r in BUSINESS_PROFILE for r in rules) else "NO ❌")

# The discount helper that C4's price tool will reuse (discounts never stack: the bigger one wins):
print("Donor campaign for a small NGO:", discounted_price(PRICES["campaigns"]["donor_outreach_campaign"], small_ngo=True), "USD")

# What a router hint looks like at the end of the prompt:
print("\n" + build_system_prompt("The user is interested. Collect their name and email.")[-120:])

## 🧭 Router

Before answering, the agent **routes** each message: it decides what kind of message it is, so each kind gets the right handling.

| Intent | Example | How the agent handles it |
|---|---|---|
| `general_question` | "How much is a donor campaign for a small NGO?" | Answer from the business info; unknown or not offered → `record_feedback` |
| `lead_collection` | "We'd like to book a volunteer drive", "maya@greensteps.org" | Collect missing details → `record_customer_interest` |
| `feedback` | "Leila's emails were wonderful!" | Thank them → `record_feedback` |

`classify_intent(message, history, client, model)` in [`router.py`](router.py):
1. asks the LLM to classify the message at **temperature 0** (same input → same answer) and to reply with **JSON only**:
   `{"intent": ..., "confidence": ..., "reason": ...}`
2. falls back to a **keyword classifier** if there is no LLM, the API call fails, or the reply isn't valid JSON, so routing never crashes the chat.

It sees the last few messages, so a short reply like *"maya@greensteps.org"* after Rima asked for an email still counts as `lead_collection`.
Every decision is printed as `[ROUTER] ...` and saved to `logs/router.jsonl`.

The route then changes the agent's behavior: `ROUTE_HINTS[intent]` is added to the end of the system prompt for that message (`build_system_prompt(route_hint)` from the previous section).

In [ ]:
from router import INTENTS, ROUTE_HINTS, classify_intent, keyword_classify

print("Intents:", INTENTS)
for intent, hint in ROUTE_HINTS.items():
    print(f"\n{intent}:\n  {hint}")

In [ ]:
# Try the router on a few messages. We have no LLM client yet (it's created in the next section),
# so this uses the keyword fallback. In the agent loop the LLM router is used.
ROUTER_EXAMPLES = [
    "What are your opening hours?",
    "How much is a donor campaign for a small NGO?",
    "We'd like to book a volunteer drive for March",
    "Leila's emails were wonderful, thank you!",
    "The results report came two weeks late, I'm disappointed.",
    "Can you write our EU grant proposal?",
]
for text in ROUTER_EXAMPLES:
    print(f"\n> {text}")
    classify_intent(text)

# A short reply only makes sense with the conversation before it:
history = [
    {"role": "user", "content": "We need a partnership pitch pack before the summer."},
    {"role": "assistant", "content": "Happy to help! What's your name and email so our team can schedule the discovery call?"},
]
print("\n> Maya, maya@greensteps.org")
classify_intent("Maya, maya@greensteps.org", history)

## 🤖 LLM Client + Agent Loop

### The LLM client
[`llm.py`](llm.py) reads `.env` and creates an `openai` client:

| `.env` setting | Provider | Default model |
|---|---|---|
| `LLM_PROVIDER=gemini` (default) | Google Gemini via its OpenAI-compatible endpoint | `gemini-2.5-flash` |
| `LLM_PROVIDER=openai` | OpenAI | `gpt-4o-mini` |

`LLM_MODEL` overrides the model. The key is read from `.env` only. If it's missing, we get a clear message, not a crash.

### The agent loop
`run_agent(...)` in [`agent.py`](agent.py) handles one user message:

1. **Route**: `classify_intent()` picks the intent; its hint goes into the system prompt.
2. **Build the messages**: system prompt + **chat history** (short-term memory, the last ~20 turns) + the new message.
3. **Tool loop**: call the model with `tools=OPENAI_TOOLS`.
   - If the reply contains **tool calls**, run each one with `run_tool()`, append the results as `"role": "tool"` messages, and call the model again.
   - If the reply is **plain text**, that's the answer.
   - At most 5 rounds, so a confused model can't loop forever.
4. **Reflect** (optional, next section) and return the reply.

Problems like a wrong API key, no internet or a rate limit become a friendly message such as *"⚠️ The AI service is busy right now..."*.

> **Short-term memory:** the model itself remembers nothing between calls. We resend the conversation every time,
> which is how Rima knows "it" means the volunteer drive mentioned two messages ago.

In [ ]:
from llm import LLMConfigError, create_client

try:
    client, MODEL = create_client()
    print(f"✅ LLM client ready (model: {MODEL})")
except LLMConfigError as e:
    client, MODEL = None, None
    print(f"⚠️ {e}")
    print("The notebook still runs, but Rima will reply with a setup message until a key is added.")

In [ ]:
from agent import run_agent

USE_REFLECTION = False  # switched on in the Reflection section below
reflect = None          # the reflection function (also created below)


def chat(message, history):
    """
    One chat turn. This is the function Gradio's ChatInterface calls.

    message: the user's new message (text)
    history: earlier messages as [{"role": "user" | "assistant", "content": "..."}]
    returns: Rima's reply (text)
    """
    result = run_agent(
        message,
        history,
        client,
        MODEL,
        build_system_prompt,
        reflect=reflect if USE_REFLECTION else None,
    )
    return result["reply"]

In [ ]:
# A short scripted conversation covering all three routes and both tools.
# Watch the [ROUTER] and [TOOL] lines: they show the agent's decisions.
def demo_conversation(user_messages):
    history = []
    for text in user_messages:
        print(f"\n👤 User: {text}")
        reply = chat(text, history)
        print(f"🤝 Rima: {reply}")
        history += [{"role": "user", "content": text}, {"role": "assistant", "content": reply}]
    return history


demo_history = demo_conversation([
    "Hi! How much is a donor campaign for a small NGO?",                          # general_question (discount: 600 - 20% = 480)
    "Can you write our EU grant proposal?",                                       # not offered -> record_feedback
    "I'm Maya from GreenSteps, maya@greensteps.org, we need volunteers in March",  # lead_collection -> record_customer_interest
    "By the way, Leila's emails for our last campaign were wonderful!",           # feedback -> record_feedback
])

## 🪞 Reflection (bonus)

**Reflection** means the agent checks its own work before answering.
After the agent loop writes a **draft**, a second model call acts as a reviewer. It reads:

- Rima's **instructions** and the **business information** (summary + PDF),
- the user's message and the **tool calls** made for it,
- the **draft** reply,

and checks it against six points: only real facts, prices and discounts; no promised fundraising results; no confirmed bookings or dates;
no claim that something was recorded unless the tool actually recorded it; answers the question in Rima's voice;
and under ~120 words unless listing prices.

The reviewer replies with JSON: `{"ok": true}` keeps the draft, and `{"ok": false, "problems": [...], "revised": "..."}` replaces it.
Every check is printed as `[REFLECTION] approved / revised / skipped` and logged to `logs/reflection.jsonl` with both the **draft** and the **final** reply.

If the reviewer fails (API error, bad JSON), the original draft is used, so reflection can only improve a reply, never break the chat.
It costs one extra model call per message, so it can be switched off with `USE_REFLECTION = False`.

In [ ]:
from reflection import make_reflector

# The reviewer gets the instructions and the business facts (but not the persona: it reviews, it doesn't chat).
reflect = make_reflector(client, MODEL, INSTRUCTIONS, BUSINESS_SUMMARY + "\n\n" + BUSINESS_PROFILE) if client else None
USE_REFLECTION = reflect is not None  # chat() now reflects on every reply
print("Reflection is", "ON" if USE_REFLECTION else "OFF (no LLM client)")

In [ ]:
# Show reflection fixing a deliberately bad draft: it promises a fundraising result (never allowed),
# confirms a start date, and uses a wrong price (the donor outreach campaign is $600, not $500).
bad_draft = ("Great news! We'll raise $20,000 for you. The donor outreach campaign is $500 and "
             "we start next Monday, consider it booked!")
if reflect:
    print("DRAFT: ", bad_draft)
    print("FINAL: ", reflect("We're a small NGO, how much is a donor campaign and what can we expect?", bad_draft))
else:
    print("Add an API key to .env to see reflection in action.")

## 💬 Gradio Demo

[Gradio](https://www.gradio.app/)'s `ChatInterface` turns our `chat(message, history)` function into a chat web page:
Gradio keeps the history and passes it in on every message (in the same `{"role", "content"}` format the agent uses).

Run the cell below, then chat in the window that appears (or open the local URL it prints).
Watch this notebook's output while you chat to see the `[ROUTER]`, `[TOOL]` and `[REFLECTION]` lines,
and check `logs/` for the saved leads and feedback.

Try:
- *"How much is a donor campaign for a small NGO?"* (general question, discount rule → 480 USD)
- *"Can you write our EU grant proposal?"* (not offered → `record_feedback`)
- *"I'm Maya from GreenSteps, maya@greensteps.org, we need volunteers in March"* (→ `record_customer_interest`)
- *"Leila's emails were wonderful, thank you!"* (feedback → `record_feedback`)

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chat,
    title="🤝 Bridgeway – Chat with Rima",
    description=(
        "Rima is the outreach advisor of Bridgeway – Fundraising & Communications Consultancy for NGOs (Hamra, Beirut). "
        "Ask about donor outreach, partnership pitches, volunteer drives, grant tracking, prices and discounts, "
        "or leave your details and our team will schedule a free discovery call."
    ),
    examples=[
        "How much is a donor campaign for a small NGO?",
        "What's included in the partnership pitch pack?",
        "Can you write our EU grant proposal?",
        "I'm Maya from GreenSteps, maya@greensteps.org, we need volunteers in March",
        "Leila's emails were wonderful, thank you!",
    ],
)
demo.launch()